### 1. 라이브러리 import 및 버전 확인

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torchinfo import summary
import matplotlib.pyplot as plt

print(torch.__version__)
print(torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

2.7.1+cu118
0.22.1+cu118
Using device: cuda


### 2. BasicBlock & BottleneckBlock 정의


In [3]:
class BasicBlock(nn.Module):
    """ResNet-34용 블록 (3x3 conv x 2)"""
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, use_skip=True):
        super().__init__()
        self.use_skip = use_skip

        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if use_skip and (stride != 1 or in_channels != out_channels):
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_skip:
            out += self.shortcut(x)
        return self.relu(out)


class BottleneckBlock(nn.Module):
    """ResNet-50용 블록 (1x1 → 3x3 → 1x1)"""
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1, use_skip=True):
        super().__init__()
        self.use_skip = use_skip

        self.conv1 = nn.Conv2d(in_channels, out_channels,
                               kernel_size=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels,
                               kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion,
                               kernel_size=1, bias=False)
        self.bn3   = nn.BatchNorm2d(out_channels * self.expansion)

        self.relu  = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if use_skip and (stride != 1 or in_channels != out_channels * self.expansion):
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.use_skip:
            out += self.shortcut(x)
        return self.relu(out)

### 3. build_resnet 함수 정의

In [4]:
CONFIGS = {
    "resnet34": {
        "block": BasicBlock,
        "layers": [3, 4, 6, 3],
        "channels": [64, 128, 256, 512],
    },
    "resnet50": {
        "block": BottleneckBlock,
        "layers": [3, 4, 6, 3],
        "channels": [64, 128, 256, 512],
    },
}

def build_resnet(model_name="resnet50", num_classes=37, use_skip=True):
    cfg      = CONFIGS[model_name]
    block    = cfg["block"]
    layers   = cfg["layers"]
    channels = cfg["channels"]

    net_layers = [
        nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    ]

    in_channels = 64

    for i, (num_blocks, out_channels) in enumerate(zip(layers, channels)):
        stride = 1 if i == 0 else 2

        net_layers.append(block(in_channels, out_channels,
                                stride=stride, use_skip=use_skip))
        in_channels = out_channels * block.expansion

        for _ in range(1, num_blocks):
            net_layers.append(block(in_channels, out_channels,
                                    stride=1, use_skip=use_skip))

    net_layers += [
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(in_channels, num_classes),
    ]

    return nn.Sequential(*net_layers)

### 4. 모델 생성 및 구조 확인

In [5]:
resnet34   = build_resnet("resnet34", use_skip=True)
plainnet34 = build_resnet("resnet34", use_skip=False)
resnet50   = build_resnet("resnet50", use_skip=True)
plainnet50 = build_resnet("resnet50", use_skip=False)

print("=== ResNet-50 구조 ===")
summary(resnet50, input_size=(1, 3, 224, 224))

=== ResNet-50 구조 ===


Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [1, 37]                   --
├─Conv2d: 1-1                            [1, 64, 112, 112]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 112, 112]         128
├─ReLU: 1-3                              [1, 64, 112, 112]         --
├─MaxPool2d: 1-4                         [1, 64, 56, 56]           --
├─BottleneckBlock: 1-5                   [1, 256, 56, 56]          --
│    └─Conv2d: 2-1                       [1, 64, 56, 56]           4,096
│    └─BatchNorm2d: 2-2                  [1, 64, 56, 56]           128
│    └─ReLU: 2-3                         [1, 64, 56, 56]           --
│    └─Conv2d: 2-4                       [1, 64, 56, 56]           36,864
│    └─BatchNorm2d: 2-5                  [1, 64, 56, 56]           128
│    └─ReLU: 2-6                         [1, 64, 56, 56]           --
│    └─Conv2d: 2-7                       [1, 256, 56, 56]          16,38

### 5. 데이터셋 로딩

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.OxfordIIITPet(
    root='./data', split='trainval',
    download=True, transform=transform
)
test_dataset = datasets.OxfordIIITPet(
    root='./data', split='test',
    download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    num_workers=0  # ✅ 0으로 변경 (멀티프로세싱 비활성화)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    num_workers=0  # ✅ 0으로 변경
)

print(f"Train: {len(train_dataset)}장, Test: {len(test_dataset)}장")
print(f"클래스 수: {len(train_dataset.classes)}")

Train: 3680장, Test: 3669장
클래스 수: 37


### 6. train_model 함수 정의

In [10]:
def train_model(model, train_loader, test_loader, epochs=5, lr=0.001):
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [],
               "train_acc": [],  "val_acc": []}

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss, correct, total = 0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)

        history["train_loss"].append(total_loss / len(train_loader))
        history["train_acc"].append(correct / total)

        # --- Validation ---
        model.eval()
        total_loss, correct, total = 0, 0, 0

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss    = criterion(outputs, labels)

                total_loss += loss.item()
                correct    += (outputs.argmax(1) == labels).sum().item()
                total      += labels.size(0)

        history["val_loss"].append(total_loss / len(test_loader))
        history["val_acc"].append(correct / total)

        print(f"[Epoch {epoch+1}/{epochs}] "
              f"Train Acc: {history['train_acc'][-1]:.4f} | "
              f"Val Acc: {history['val_acc'][-1]:.4f}")

    return history

### 7. 학습 실행

In [14]:
# Bottleneck (ResNet-50)
print("=== ResNet-50 학습 ===")
h_resnet50 = train_model(resnet50, train_loader, test_loader, epochs=15)

print("\n=== PlainNet-50 학습 ===")
h_plain50 = train_model(plainnet50, train_loader, test_loader, epochs=15)

# BasicBlock (ResNet-34)
print("\n=== ResNet-34 학습 ===")
h_resnet34 = train_model(resnet34, train_loader, test_loader, epochs=15)

print("\n=== PlainNet-34 학습 ===")
h_plain34 = train_model(plainnet34, train_loader, test_loader, epochs=15)

=== ResNet-50 학습 ===
[Epoch 1/15] Train Acc: 0.1992 | Val Acc: 0.1840
[Epoch 2/15] Train Acc: 0.2505 | Val Acc: 0.1927
[Epoch 3/15] Train Acc: 0.2595 | Val Acc: 0.2080
[Epoch 4/15] Train Acc: 0.2984 | Val Acc: 0.2069
[Epoch 5/15] Train Acc: 0.3187 | Val Acc: 0.2003
[Epoch 6/15] Train Acc: 0.3427 | Val Acc: 0.2224
[Epoch 7/15] Train Acc: 0.3859 | Val Acc: 0.2189
[Epoch 8/15] Train Acc: 0.4043 | Val Acc: 0.2074
[Epoch 9/15] Train Acc: 0.4533 | Val Acc: 0.2393
[Epoch 10/15] Train Acc: 0.4720 | Val Acc: 0.2333
[Epoch 11/15] Train Acc: 0.5443 | Val Acc: 0.2339
[Epoch 12/15] Train Acc: 0.5856 | Val Acc: 0.2418
[Epoch 13/15] Train Acc: 0.6413 | Val Acc: 0.2398
[Epoch 14/15] Train Acc: 0.6641 | Val Acc: 0.2341
[Epoch 15/15] Train Acc: 0.7084 | Val Acc: 0.2276

=== PlainNet-50 학습 ===
[Epoch 1/15] Train Acc: 0.0429 | Val Acc: 0.0324
[Epoch 2/15] Train Acc: 0.0486 | Val Acc: 0.0480
[Epoch 3/15] Train Acc: 0.0476 | Val Acc: 0.0551
[Epoch 4/15] Train Acc: 0.0476 | Val Acc: 0.0477
[Epoch 5/15] Train

### 8. 결과표

In [15]:
import pandas as pd

data = {
    "plain": [
        round(max(h_plain34["val_acc"]) * 100, 2),
        round(max(h_plain50["val_acc"]) * 100, 2),
    ],
    "ResNet": [
        round(max(h_resnet34["val_acc"]) * 100, 2),
        round(max(h_resnet50["val_acc"]) * 100, 2),
    ],
}

df = pd.DataFrame(data, index=["34 layers", "50 layers"])

df.style\
  .format("{:.2f}")\
  .set_properties(**{"text-align": "center"})\
  .set_table_styles([
      {"selector": "th", "props": [("text-align", "center")]},
      {"selector": "thead tr th", "props": [("border-top", "2px solid black"),
                                             ("border-bottom", "1px solid black")]},
      {"selector": "tbody tr:last-child td", "props": [("border-bottom", "2px solid black")]},
  ])

,plain,ResNet
34 layers,8.83,28.10
50 layers,7.44,24.18
